# Configuracion Basica para usar Spark Sql

In [1]:
# Configurar Java y el entorno de Spark antes de importar PySpark

import os
import sys
from pathlib import Path

CONDA_PREFIX = Path(sys.prefix)

# Java instalado dentro del entorno de Anaconda
java_home = CONDA_PREFIX / "Library"

if not (java_home / "bin" / "java.exe").exists():
    raise FileNotFoundError(
        "No se encontró Java dentro del entorno de Anaconda.\n"
        "Abre Anaconda Prompt, activa el entorno spark y ejecuta:\n"
        "conda install -c conda-forge openjdk=8"
    )

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

#Hadoop
HADOOP_HOME = r"C:\hadoop"

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["PATH"] = (
    os.path.join(HADOOP_HOME, "bin")
    + os.pathsep
    + os.environ.get("PATH", "")
)

print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("Hadoop bin:", os.path.join(HADOOP_HOME, "bin"))

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

HADOOP_HOME: C:\hadoop
Hadoop bin: C:\hadoop\bin
Python: c:\Users\RyanHz\anaconda3\envs\spark\python.exe
JAVA_HOME: c:\Users\RyanHz\anaconda3\envs\spark\Library


In [2]:
# Crear la sesión de Spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HousingSparkSQL")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

print("Spark iniciado correctamente")
print("Versión de Spark:", spark.version)

Spark iniciado correctamente
Versión de Spark: 3.5.6


In [3]:
# Localizar el archivo CSV

from pathlib import Path

posibles_archivos = [
    Path("../../../../Archivos-Analisis/files-tarea-m33/vgsales.csv")
]

ruta_csv = next((ruta for ruta in posibles_archivos if ruta.exists()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró el CSV. Coloca 'vgsales.csv' "
        "en la misma carpeta que este notebook."
    )

print("Archivo encontrado:", ruta_csv.resolve())

Archivo encontrado: C:\Users\RyanHz\Documents\EBAC\VS\Archivos-Analisis\files-tarea-m33\vgsales.csv


# Contenido

In [4]:
vgsales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

In [5]:
vgsales_df.createOrReplaceTempView('VGSales')

sql_str = "select Publisher, sum(NA_Sales), sum(Global_Sales) from VGSales group by Genre, Publisher order by Publisher desc"
spark.sql(sql_str).show()

+--------------------+-------------------+------------------+
|           Publisher|      sum(NA_Sales)| sum(Global_Sales)|
+--------------------+-------------------+------------------+
|        responDESIGN|0.09000000000000001|              0.13|
|           mixi, Inc|                0.0|              0.86|
|inXile Entertainment|               0.02|               0.1|
|     imageepoch Inc.|                0.0|              0.01|
|     imageepoch Inc.|                0.0|              0.03|
|         id Software|               0.02|              0.03|
|                iWin|                0.0|              0.06|
|              fonfun|                0.0|              0.02|
|     dramatic create|                0.0|               0.1|
|     dramatic create|                0.0|              0.01|
|   bitComposer Games|                0.0|              0.03|
|   bitComposer Games|               0.16|              0.38|
|         Zushi Games|               0.04|              0.05|
|       

In [36]:
# Redondeo de las columnas
sql_str = "select Publisher, round(sum(NA_Sales), 2), round(sum(Global_Sales), 2) from VGSales group by Genre, Publisher order by Publisher desc"
spark.sql(sql_str).show()

+--------------------+-----------------------+---------------------------+
|           Publisher|round(sum(NA_Sales), 2)|round(sum(Global_Sales), 2)|
+--------------------+-----------------------+---------------------------+
|        responDESIGN|                   0.09|                       0.13|
|           mixi, Inc|                    0.0|                       0.86|
|inXile Entertainment|                   0.02|                        0.1|
|     imageepoch Inc.|                    0.0|                       0.01|
|     imageepoch Inc.|                    0.0|                       0.03|
|         id Software|                   0.02|                       0.03|
|                iWin|                    0.0|                       0.06|
|              fonfun|                    0.0|                       0.02|
|     dramatic create|                    0.0|                        0.1|
|     dramatic create|                    0.0|                       0.01|
|   bitComposer Games|   

In [37]:
spark.sql(sql_str).show(10, 40)
#         Numero de datos | numero de caracteres

+--------------------+-----------------------+---------------------------+
|           Publisher|round(sum(NA_Sales), 2)|round(sum(Global_Sales), 2)|
+--------------------+-----------------------+---------------------------+
|        responDESIGN|                   0.09|                       0.13|
|           mixi, Inc|                    0.0|                       0.86|
|inXile Entertainment|                   0.02|                        0.1|
|     imageepoch Inc.|                    0.0|                       0.01|
|     imageepoch Inc.|                    0.0|                       0.03|
|         id Software|                   0.02|                       0.03|
|                iWin|                    0.0|                       0.06|
|              fonfun|                    0.0|                       0.02|
|     dramatic create|                    0.0|                        0.1|
|     dramatic create|                    0.0|                       0.01|
+--------------------+---

In [38]:
# Se puede obtener el plan de ejecucion con explain

spark.sql(sql_str).explain()

# Plan extendido
# spark.sql(sql_str).explain(extended= True)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [Publisher#22 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(Publisher#22 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=218]
      +- HashAggregate(keys=[Genre#21, Publisher#22], functions=[sum(NA_Sales#23), sum(Global_Sales#27)])
         +- Exchange hashpartitioning(Genre#21, Publisher#22, 200), ENSURE_REQUIREMENTS, [plan_id=215]
            +- HashAggregate(keys=[Genre#21, Publisher#22], functions=[partial_sum(NA_Sales#23), partial_sum(Global_Sales#27)])
               +- FileScan csv [Genre#21,Publisher#22,NA_Sales#23,Global_Sales#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/RyanHz/Documents/EBAC/VS/Archivos-Analisis/files-tarea-..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Genre:string,Publisher:string,NA_Sales:double,Global_Sales:double>




In [39]:
spark.sql(sql_str).columns

['Publisher', 'round(sum(NA_Sales), 2)', 'round(sum(Global_Sales), 2)']

# Particion de Datos

In [40]:
# Implementacion de Partitionby
spark = SparkSession.builder.appName('Partitionby() PySpark').getOrCreate()

# Lee en el df el archivo
df = spark.read.option('header', True).csv('../../../../Archivos-Analisis/files-tarea-m33/vgsales.csv')

# Impriome el esquema
df.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: string (nullable = true)
 |-- EU_Sales: string (nullable = true)
 |-- JP_Sales: string (nullable = true)
 |-- Other_Sales: string (nullable = true)
 |-- Global_Sales: string (nullable = true)



## partition by
Generando pedazos de informacion

In [ ]:
import os

HADOOP_HOME = r"C:\hadoop"
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["PATH"] = os.path.join(HADOOP_HOME, "bin") + os.pathsep + os.environ.get("PATH", "")

winutils = os.path.join(HADOOP_HOME, "bin", "winutils.exe")
print("HADOOP_HOME:", HADOOP_HOME)
print("winutils path:", winutils)
print("exists:", os.path.exists(winutils))

if not os.path.exists(winutils):
    raise FileNotFoundError(
        f"No se encontró winutils.exe en {winutils}. "
        "Descarga un winutils compatible y colócalo en HADOOP_HOME\\bin, luego reinicia el kernel."
    )

Py4JJavaError: An error occurred while calling o87.csv.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(FileOut

In [ ]:
df.write.option('header', True).partitionBy('Platform').mode('overwrite').csv('../../../../Archivos-Analisis/files-tarea-m33/partition/year')

## Coalesce y Repartition

In [38]:
# Implementacion de Partitionby
spark = SparkSession.builder.appName('coalesce() PySpark').getOrCreate()

# Lee en el df el archivo
df = spark.read.option('header', True).csv('../../../../Archivos-Analisis/files-tarea-m33/vgsales.csv')

# Impriome el esquema
df.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: string (nullable = true)
 |-- EU_Sales: string (nullable = true)
 |-- JP_Sales: string (nullable = true)
 |-- Other_Sales: string (nullable = true)
 |-- Global_Sales: string (nullable = true)



In [ ]:
# Genera en el directorio el numero de archivos solicitados
df.repartition(20).write.mode('overwrite').option('header', True).csv('../../../../Archivos-Analisis/files-tarea-m33/partition/rep')

In [40]:
# Usando coalesce
df2 = df.repartition(20)
df2.rdd.getNumPartitions()

20

In [41]:
df3 = df2.coalesce(10)

In [ ]:
df3.write.mode('overwrite').option('header', True).csv('../../../../Archivos-Analisis/files-tarea-m33/partition/coalesce')

## PyArrow

In [8]:
from pyarrow import csv
import pyarrow as pa

In [18]:
archivo = '../../../../Archivos-Analisis/files-tarea-m33/vgsales.csv'
tab_vgsales = csv.read_csv(archivo)

In [10]:
# Estructura de vgsales visto desde Arrow
tab_vgsales

pyarrow.Table
Rank: int64
Name: string
Platform: string
Year: int64
Genre: string
Publisher: string
NA_Sales: double
EU_Sales: double
JP_Sales: double
Other_Sales: double
Global_Sales: double
----
Rank: [[1,2,3,4,5,...,12815,12816,12817,12818,12819],[12820,12821,12822,12823,12824,...,16596,16597,16598,16599,16600]]
Name: [["Wii Sports","Super Mario Bros.","Mario Kart Wii","Wii Sports Resort","Pokemon Red/Pokemon Blue",...,"SeaBlade","Jikkyou Powerful Pro Yakyuu DreamCast Edition","Carmen Sandiego: The Secret of the Stolen Drums","Tokyo Mono Harashi: Karasu no Mori Gakuen Kitan","Activision Hits: Remixed"],["Death Jr. II: Root of Evil","Hail to the Chimp","MotoGP 4 - Official Game of MotoGP","Legend of Kay","Kyoukai Senjou no Horizon Portable",...,"Woody Woodpecker in Crazy Castle 5","Men in Black II: Alien Escape","SCORE International Baja 1000: The Official Game","Know How 2","Spirits & Spells"]]
Platform: [["Wii","NES","Wii","Wii","GB",...,"XB","DC","PS2","PSP","PSP"],["PSP","X360","

In [11]:
tab_vgsales.column_names

['Rank',
 'Name',
 'Platform',
 'Year',
 'Genre',
 'Publisher',
 'NA_Sales',
 'EU_Sales',
 'JP_Sales',
 'Other_Sales',
 'Global_Sales']

In [12]:
tab_vgsales.schema

Rank: int64
Name: string
Platform: string
Year: int64
Genre: string
Publisher: string
NA_Sales: double
EU_Sales: double
JP_Sales: double
Other_Sales: double
Global_Sales: double

In [13]:
tab_vgsales.num_columns

11

In [14]:
# Pasa la estructura a Panas
df = tab_vgsales.to_pandas()

In [15]:
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [20]:
# Agrupacion de la tabla x Genero
tab_genre = tab_vgsales.group_by('Genre').aggregate([('NA_Sales', 'sum')])

In [23]:
df1 = tab_genre.to_pandas()
df1.head()

,Genre,NA_Sales_sum
0,Sports,683.35
1,Platform,447.05
2,Racing,359.42
3,Role-Playing,327.28
4,Puzzle,123.78


In [25]:
# Agregar columnas con append_column
tab2 = tab_vgsales.append_column('Test', pa.array(['0'] * len(tab_vgsales), pa.string()))
df2 = tab2.to_pandas()
df2.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Test
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,0
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,0
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,0
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,0
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,0


## Uso de Parquet

In [28]:
import pyarrow.parquet as pq
import pandas as pd
import pyarrow as pa

df = pd.DataFrame({'lin1': [-20, 100, 200],
                   'lin2': ['este', 'oeste', 'norte'],
                   'lin3': [False, True, True]},
                   index = list('abc'))

In [29]:
df

,lin1,lin2,lin3
a,-20,este,False
b,100,oeste,True
c,200,norte,True


In [30]:
tab_example = pa.Table.from_pandas(df)
pq.write_table(tab_example, 'ejemplo.parquet')

In [34]:
# Leer el archivo tipo parquet
tab2 = pq.read_table('ejemplo.parquet')
tab2.to_pandas()

,lin1,lin2,lin3
a,-20,este,False
b,100,oeste,True
c,200,norte,True


In [35]:
tab2

pyarrow.Table
lin1: int64
lin2: large_string
lin3: bool
__index_level_0__: large_string
----
lin1: [[-20,100,200]]
lin2: [["este","oeste","norte"]]
lin3: [[false,true,true]]
__index_level_0__: [["a","b","c"]]